## Near field energy transfer calculation in highly parallelized setting

### (1) Introduction
The overell framework this script follows is the same as the previous script#2 on computation of the energy transfer matrix elements. However, here it it done in a highly parallel way where the matrix elements are computed parallely using mpi4py.


#### Define Emitter Data

In [4]:
import os, sys
cwd = os.getcwd()
from pyRET import *
consts = Consts()

#%% Define emitters
emitters_savefile = os.path.join(cwd, "Emitters_data.json")
consts = Consts()

Em_data = Emitters_data()


#Emitter A:
em = "A"
Em_data.EmType[em] = em
Em_data.nIndex[em] = nIndex = 2.4
Em_data.T1[em] = {}
Em_data.w[em] = {}
Em_data.k[em] = {}
Em_data.pMPs[em] = {}
transitions = ["EDx","EDy","EDz","MDx","MDy","MDz"]
for tran in transitions:
    
    Em_data.T1[em][tran] = 1e-6
    Em_data.w[em][tran] = 1.9396*consts.qe/consts.hbar-1j/Em_data.T1[em][tran]

    Em_data.k[em][tran] = nIndex*Em_data.w[em][tran]/consts.c

    pEDA =  getED(Em_data.T1[em][tran], np.real(Em_data.w[em][tran]),nIndex)
    mMDA =  getMD(Em_data.T1[em][tran], np.real(Em_data.w[em][tran]),nIndex)    
    
    
    nx = np.array([1, 0, 0], dtype = complex)
    ny = np.array([0, 1, 0], dtype = complex)
    nz = np.array([0, 0, 1], dtype = complex)

    
    Em_data.pMPs[em][tran] = {}
    Em_data.pMPs[em][tran][1] = {}
    
    Parity = -1 if "ED" in tran else 1 #True for only dipole like modes
    Pval = pEDA if "ED" in tran else mMDA
    if "x" in tran: n = nx
    if "y" in tran: n = ny
    if "z" in tran: n = nz
    
    
    Em_data.pMPs[em][tran][1][Parity] = Pval*n
    


#Emitter B(NV):
em = "NV"
iKSs = [ 
    87, 122, 123, 126, 127, 128,
       ]
stateis = [f"{iKS}" for iKS in iKSs]
statefs = stateis
Em_data.EmType[em] = em
Em_data.T1[em] = {}
Em_data.w[em] = {}
Em_data.k[em] = {}
Em_data.pMPs[em] = {}
Em_data.nIndex[em] = nIndex = 2.4

#set the transitions energies
#etc.etc..

for statei in stateis:
    for statef in statefs:
        
        tran = f"{statei}-to-{statef}"
        Em_data.pMPs[em][tran] = {}

        Em_data.T1[em][tran] = -1
        Em_data.w[em][tran] = 0*consts.qe/consts.hbar

        Em_data.k[em][tran] = 0;#nIndex*Em_data.w[em][tran]/consts.c
        Em_data.pMPs[em][tran] = {}




with open(emitters_savefile, "w") as f:
    json.dump(Em_data.Encode(), f)

#### Emission Matrix Element Calculations: Creating the v.in files

In [5]:
#%% Calculation of emission matrix elements of the NV transitions
path = os.path.join(cwd, "Emission_Matrix_Elements")
if not os.path.exists(path):
    os.makedirs(path)


#Create dummy position data for this.
Pos_data = Positions_data()

#Positions
RABs = np.array([10e-9,]) 
TABs = np.array([0.]) 
PABs = np.array([0.]) 

RABg, TABg, PABg = np.meshgrid(RABs, TABs, PABs)
RABvects = pol2cart(RABg, TABg, PABg, np.zeros(3))

#Define and save Posiotions data class:
Pos_data.Rs["NV"] = {}
Pos_data.Rs["NV"]["A"] = RABs
Pos_data.Ps["NV"] = {}
Pos_data.Ps["NV"]["A"] = PABs
Pos_data.Ts["NV"] = {}
Pos_data.Ts["NV"]["A"] = TABs
Pos_data.Rg["NV"] = {}
Pos_data.Rg["NV"]["A"] = RABg
Pos_data.Pg["NV"] = {}
Pos_data.Pg["NV"]["A"] = PABg
Pos_data.Tg["NV"] = {}
Pos_data.Tg["NV"]["A"] = TABg
Pos_data.Rvects["NV"] = {}
Pos_data.Rvects["NV"]["A"] = RABvects
    

with open(f"{path}/Posdata.json", "w") as f:
    json.dump(Pos_data.Encode(), f)


#Create v.in file for matrix element calculation
absorber_id = "NV"
emitter_id = "NV"


photon_energy_eV = 1.9396
w = photon_energy_eV*consts.qe/consts.hbar
ws = np.array([w])
Lmax = 2

absorber_transitions = []
for iiKS, iKS in enumerate(iKSs):
    for ijKS, jKS in enumerate(iKSs):
        if iKS >= jKS: continue
        absorber_transitions.append(f"{iKS}-to-{jKS}")


wfdatapath = os.path.join(
    os.path.dirname(cwd),      
    "load_wavefunctions",
    "Extracted_Wavefunctions",
    "NV"
    )
wfdatafiles = [f"{wfdatapath}/{state}.json" for state in [f"{iKS}" for iKS in iKSs]]
Emdatafile = os.path.join(cwd, "Emitters_data.json")
Posdatafile = os.path.join(cwd, "Positions_data.json")
for absorber_transition in absorber_transitions:

    vin = Vin(
        emdatafile = Emdatafile, 
        posdatafile = Posdatafile, 
        wfdatafiles = wfdatafiles,
        absorber_id=absorber_id, 
        emitter_id= emitter_id, 
        absorber_transitions =[absorber_transition],
        nIndex = Em_data.nIndex[emitter_id], Lmax = Lmax,
        ws = ws, savepath = "./",savefile = "V.json",
        radtype_abs = "J")
        
    vpath = os.path.join(cwd,
        f"Absorber_{absorber_id}",
        f"Transition_{absorber_transition}",
        f"Emitter_{emitter_id}_{absorber_transition}"
    )
    if not os.path.exists(vpath):
        os.makedirs(vpath)
    
    vin.write_file(f"{vpath}/v.in")


    print(f"Creating v.in for ")
    print(f"\tabsorber: {absorber_id}, emitter: {emitter_id}")
    print(f"\tabsorber transition: {absorber_transition}")
    print(f"\tAt photon energies {ws*consts.hbar/consts.qe} eV")
    print(f"\tMax multipole angular momentum = {Lmax}")
    print(f"\tSaved at {vpath}/v.in")    

Creating v.in for 
	absorber: NV, emitter: NV
	absorber transition: 87-to-122
	At photon energies [1.9396] eV
	Max multipole angular momentum = 2
	Saved at /pscratch/sd/c/chattara/github_testing/fresh1/py-RET-code-development/Examples/compute_matrix_elements_parallel/Absorber_NV/Transition_87-to-122/Emitter_NV_87-to-122/v.in
Creating v.in for 
	absorber: NV, emitter: NV
	absorber transition: 87-to-123
	At photon energies [1.9396] eV
	Max multipole angular momentum = 2
	Saved at /pscratch/sd/c/chattara/github_testing/fresh1/py-RET-code-development/Examples/compute_matrix_elements_parallel/Absorber_NV/Transition_87-to-123/Emitter_NV_87-to-123/v.in
Creating v.in for 
	absorber: NV, emitter: NV
	absorber transition: 87-to-126
	At photon energies [1.9396] eV
	Max multipole angular momentum = 2
	Saved at /pscratch/sd/c/chattara/github_testing/fresh1/py-RET-code-development/Examples/compute_matrix_elements_parallel/Absorber_NV/Transition_87-to-126/Emitter_NV_87-to-126/v.in
Creating v.in for 


#### Absorption Matrix Element Calculations: Creating the v.in files

In [6]:
#%% Create v.in file for absorption matrix element calculation
path = os.path.join(cwd, "Absorption_Matrix_Elements")
if not os.path.exists(path):
    os.makedirs(path)

# Define Positions
phival = 0
thetaval = np.pi/2

Pos_data = Positions_data()

#Positions
RABs = 10**(np.linspace(-9, -6, 11)) #np.array([0.1, 0.5, 1, 5, 10, 20, 50])*1e-9
TABs = np.array([thetaval]) #np.linspace(0, np.pi, 3)
PABs = np.array([phival]) #np.linspace(0, np.pi, 21)

RABg, TABg, PABg = np.meshgrid(RABs, TABs, PABs)
RABvects = pol2cart(RABg, TABg, PABg, np.zeros(3))

#Define and save Posiotions data class:
Pos_data.Rs["NV"] = {}
Pos_data.Rs["NV"]["A"] = RABs
Pos_data.Ps["NV"] = {}
Pos_data.Ps["NV"]["A"] = PABs
Pos_data.Ts["NV"] = {}
Pos_data.Ts["NV"]["A"] = TABs
Pos_data.Rg["NV"] = {}
Pos_data.Rg["NV"]["A"] = RABg
Pos_data.Pg["NV"] = {}
Pos_data.Pg["NV"]["A"] = PABg
Pos_data.Tg["NV"] = {}
Pos_data.Tg["NV"]["A"] = TABg
Pos_data.Rvects["NV"] = {}
Pos_data.Rvects["NV"]["A"] = RABvects
    

with open(f"{path}/Posdata.json", "w") as f:
    json.dump(Pos_data.Encode(), f)



absorber_id = "NV"
emitter_id = "A"
emitter_transition = "EDz"
absorber_transitions = []
photon_energy_eV = 1.9396
w = photon_energy_eV*consts.qe/consts.hbar
ws = np.array([w])
Lmax = 2


for iiKS, iKS in enumerate(iKSs):
    for ijKS, jKS in enumerate(iKSs):
        if iKS >= jKS: continue
        absorber_transitions.append(f"{iKS}-to-{jKS}")
wfdatapath = os.path.join(
    os.path.dirname(cwd),
    "load_wavefunctions",
    "Extracted_Wavefunctions",
    "NV"
    )
wfdatafiles = [f"{wfdatapath}/{state}.json" for state in [f"{iKS}" for iKS in iKSs]]
Emdatafile = os.path.join(cwd,"Emitters_data.json")
Posdatafile = os.path.join(path, "Positions_data.json")
for absorber_transition in absorber_transitions:

    vin = Vin(
        emdatafile = Emdatafile, 
        posdatafile = Posdatafile, 
        wfdatafiles = wfdatafiles,
        absorber_id=absorber_id, 
        emitter_id= emitter_id, 
        absorber_transitions =[absorber_transition],
        nIndex = 2.42, Lmax = Lmax,
        ws = ws, savepath = "./",savefile = "V.json")
        
    vpath = os.path.join(
        cwd,
        "Absorption_Matrix_Elements",
        f"Absorber_{absorber_id}",
        f"Transition_{absorber_transition}",
        f"Emitter_{emitter_id}_{emitter_transition}"
    )
    if not os.path.exists(vpath):
        os.makedirs(vpath)
    
    vin.write_file(f"{vpath}/v.in")


    print(f"Creating v.in for ")
    print(f"\tabsorber: {absorber_id}, emitter: {emitter_id}")
    print(f"\tabsorber transition: {absorber_transition}")
    print(f"\tAt photon energies {ws*consts.hbar/consts.qe} eV")
    print(f"\tMax multipole angular momentum = {Lmax}")
    print(f"\tSaved at {vpath}/v.in")    
    

Creating v.in for 
	absorber: NV, emitter: A
	absorber transition: 87-to-122
	At photon energies [1.9396] eV
	Max multipole angular momentum = 2
	Saved at /pscratch/sd/c/chattara/github_testing/fresh1/py-RET-code-development/Examples/compute_matrix_elements_parallel/Absorption_Matrix_Elements/Absorber_NV/Transition_87-to-122/Emitter_A_EDz/v.in
Creating v.in for 
	absorber: NV, emitter: A
	absorber transition: 87-to-123
	At photon energies [1.9396] eV
	Max multipole angular momentum = 2
	Saved at /pscratch/sd/c/chattara/github_testing/fresh1/py-RET-code-development/Examples/compute_matrix_elements_parallel/Absorption_Matrix_Elements/Absorber_NV/Transition_87-to-123/Emitter_A_EDz/v.in
Creating v.in for 
	absorber: NV, emitter: A
	absorber transition: 87-to-126
	At photon energies [1.9396] eV
	Max multipole angular momentum = 2
	Saved at /pscratch/sd/c/chattara/github_testing/fresh1/py-RET-code-development/Examples/compute_matrix_elements_parallel/Absorption_Matrix_Elements/Absorber_NV/Tr

#### Write a Job Script to Run the CalculateV_parallel.py script with the v.in files as the inputs

In [8]:
#For advanced slurm formatting, use the pyjobtools module.

paths_to_vin_files = []
job_script_lines = []
for p, _, files in os.walk("Parallel_Computation/"):
    if "v.in" in files:
        paths_to_vin_files.append(os.path.join(p, "v.in"))
        job_script_lines.append(f"cd {os.path.abspath(p)}\n")
        job_script_lines.append(f"srun -n 20 -c 3 python -u {os.path.join(pyRETpath, 'CalculateV_parallel.py')} -in ./v.in > ./v.out\n")

with open(os.path.join(cwd, "job_script.sh"), "w") as f:
    f.writelines(job_script_lines)
    